# Phase 2 · Legal NER · Experiment 1: arch 4 (MLA) + full fine-tuning
Start from the Phase-1 **MLA language model (step 40k)**, add a **token-classification head** over BIO tags
(12 labels → 25 tags), fine-tune **all** weights, turn predicted tags back into **exact character spans**.

| Cell | What it does |
|---|---|
| 1–3 | settings, Hugging Face login, write `ner_data.py` / `ner_model.py` |
| 4 | load `Phase_2_LNER`: documents, gold spans, label list, **end-offset convention** (checked on the data) |
| 5 | document-level validation split (no public dev set) |
| 6 | **offset round-trip check**: gold spans → token tags → spans again, for two pre-tokenisations |
| 7–8 | build model, fine-tune, keep the best validation epoch |
| 9 | per-label strict P/R/F1 + error breakdown (boundary vs label vs missed vs spurious) |
| 10 | predict fold 3 → `submission_<run>.zip`, save `record.json` + the fine-tuned weights (teacher for distillation later) |

**Run:** GPU **T4 x2**, Internet **On**, Secret `HF_TOKEN`, Add Input → your dataset **`phase2-backbones`**. Save & Run All. ~1–1.5 h.

In [ ]:
# 1 · Settings
import os, sys, json, glob, time, random, zipfile, math
import numpy as np, torch

CFG = dict(
    run="p2_arch4_fullft",
    arch="arch4_mla",            # arch1_decoder | arch2_encdec | arch3_prefixlm | arch4_mla
    method="full_ft",            # this notebook: every weight is trained
    bidirectional=True,          # remove the causal mask: each token also sees the words after it
    split="fine",                # pre-tokenisation: "fine" (punctuation split off words) or "bpe" (as in pretraining); cell 6 compares them
    window=512, train_stride=256, eval_stride=384,    # tokens per window / step between windows
    batch_size=8, eval_batch=16, epochs=4,
    lr_backbone=5e-5, lr_head=1e-3, weight_decay=0.01, warmup_frac=0.06, clip=1.0, dropout=0.1,
    class_weighting="none",      # "none" or "sqrt_inv" (rare labels weigh more)
    val_frac=0.15, seed=1337, max_train_hours=3.0, log_every=50,
    smoke=False,                 # True = tiny 5-minute rehearsal (its zip is NOT a real submission)
    hf_repo="Exploration-Lab/CS-781-Capstone", out="/kaggle/working",
)
if CFG["smoke"]:
    CFG.update(run=CFG["run"] + "_smoke", epochs=1, window=256, train_stride=256, eval_stride=256, log_every=5)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
RUN_DIR = os.path.join(CFG["out"], CFG["run"]); os.makedirs(RUN_DIR, exist_ok=True)

found = glob.glob("/kaggle/input/**/phase2-backbones/manifest.json", recursive=True) or \
        glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert found, "attach the dataset phase2-backbones (Add Input -> Your Datasets)"
ROOT = os.path.dirname(found[0])
print("backbones:", ROOT, "| device", DEVICE, "x", torch.cuda.device_count(), "|", CFG["run"])

In [ ]:
# 2 · Hugging Face login (the dataset is gated)
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret("HF_TOKEN")); print("logged in (Kaggle secret)")
except Exception:
    import getpass
    login(token=os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token: ")); print("logged in")

## 3 · Phase-2 code
`ner_data.py`: parsing, character↔token alignment, BIO tags, span decoding, strict scorer (pure Python). `ner_model.py`: backbone loader for all 4 architectures, tagging head, sliding-window training and prediction.

In [ ]:
os.makedirs('p2src', exist_ok=True)

In [ ]:
%%writefile p2src/ner_data.py
# Phase 2 (L-NER) data utilities: gold parsing, character<->token alignment, BIO tags, span decoding,
# strict span scoring, submission writing. Pure Python (no torch) so it can be unit-tested anywhere.
import bisect
import json
import re
from collections import Counter, defaultdict

# ----------------------------------------------------------------------------------------------------
# 1. Gold annotations
# ----------------------------------------------------------------------------------------------------
_START = ("start", "start_char", "start_offset", "begin", "s")
_END = ("end", "end_char", "end_offset", "stop", "e")
_LABEL = ("label", "type", "entity_type", "tag", "category")
_TEXT = ("text", "entity_text", "span", "mention", "value")


def _pick(d, keys):
    for k in keys:
        if k in d:
            return d[k]
    return None


def _entity(item):
    """one annotation in any common shape -> (start, end, label, text or None)"""
    if isinstance(item, (list, tuple)):
        s, e, lab = item[0], item[1], item[2]
        txt = item[3] if len(item) > 3 else None
    elif isinstance(item, dict):
        s, e, lab, txt = _pick(item, _START), _pick(item, _END), _pick(item, _LABEL), _pick(item, _TEXT)
    else:
        raise ValueError(f"unknown annotation format: {item!r}")
    return int(s), int(e), str(lab), (txt if isinstance(txt, str) else None)


def parse_gold(raw):
    """gold json (dict doc -> list, dict doc -> {entities: [...]}, or list of records) -> {doc: [(s,e,lab,txt)]}"""
    out = {}
    if isinstance(raw, list):                                   # [{"id": .., "entities": [..]}, ...]
        for rec in raw:
            doc = str(_pick(rec, ("id", "doc_id", "document_id", "docid")))
            ents = _pick(rec, ("entities", "annotations", "spans", "labels", "gold")) or []
            out[doc] = [_entity(x) for x in ents]
        return out
    for doc, v in raw.items():
        if isinstance(v, dict):
            v = _pick(v, ("entities", "annotations", "spans", "labels", "gold")) or []
        out[str(doc)] = [_entity(x) for x in v]
    return out


def parse_texts(raw):
    """text json -> {doc: text}"""
    if isinstance(raw, list):
        return {str(_pick(r, ("id", "doc_id", "document_id", "docid"))): _pick(r, ("text", "content", "document"))
                for r in raw}
    return {str(k): (v if isinstance(v, str) else _pick(v, ("text", "content", "document"))) for k, v in raw.items()}


def end_convention(texts, gold):
    """Is `end` exclusive (text[s:e] == entity text) or inclusive (text[s:e+1])? Decided by the data."""
    exc = inc = total = 0
    bad = []
    for doc, ents in gold.items():
        t = texts.get(doc)
        if t is None:
            continue
        for s, e, lab, txt in ents:
            if txt is None:
                continue
            total += 1
            if t[s:e] == txt:
                exc += 1
            elif t[s:e + 1] == txt:
                inc += 1
            elif len(bad) < 5:
                bad.append((doc, s, e, lab, txt, t[s:e + 1]))
    conv = "inclusive" if inc > exc else "exclusive"
    return conv, dict(total_with_text=total, exclusive_match=exc, inclusive_match=inc, examples_no_match=bad)


def to_exclusive(gold, conv):
    """-> {doc: [(s, e_exclusive, label)]} (internal convention: python slice)"""
    add = 1 if conv == "inclusive" else 0
    return {d: [(s, e + add, lab) for s, e, lab, _ in ents] for d, ents in gold.items()}


# ----------------------------------------------------------------------------------------------------
# 2. Tokenisation with character offsets
# ----------------------------------------------------------------------------------------------------
# Pre-tokenisation decides where tokens can start/end. "bpe" = the split used in pretraining.
# "fine" = additionally split punctuation from words, so an entity ending in "Singh," -> "Singh" | ","
# can be cut exactly. Every chunk is still encoded by the same BPE (same vocabulary).
SPLITS = {
    ("space", "bpe"): re.compile(r" ?\S+|\s+(?!\S)|\s+"),
    ("space", "fine"): re.compile(r" ?[^\W_]+| ?_+| ?[^\w\s]|\s+(?!\S)|\s+"),
    ("a1", "bpe"): re.compile(r"\S+|\s+"),
    ("a1", "fine"): re.compile(r"[^\W_]+|_+|[^\w\s]|\s+"),
}


def tokenizer_kind(tok):
    return "space" if getattr(tok, "space_prefix", False) else "a1"


def encode_with_offsets(tok, text, split="fine"):
    """-> ids, cs, ce : token ids and each token's CORE character span [cs, ce) (leading/trailing
    whitespace removed; a whitespace-only token gets cs == ce)."""
    pat = SPLITS[(tokenizer_kind(tok), split)]
    ids, cs, ce = [], [], []
    covered = 0
    for m in pat.finditer(text):
        chunk, base = m.group(), m.start()
        assert base == covered, f"pre-tokeniser skipped characters at {covered}"
        covered = m.end()
        char_of_byte = []
        for ci, ch in enumerate(chunk):
            char_of_byte.extend([ci] * len(ch.encode("utf-8")))
        pos = 0
        for t in tok.encode_chunk(chunk.encode("utf-8")):
            n = len(tok.itos[t])
            s, e = base + char_of_byte[pos], base + char_of_byte[pos + n - 1] + 1
            pos += n
            while s < e and text[s].isspace():
                s += 1
            while e > s and text[e - 1].isspace():
                e -= 1
            ids.append(t); cs.append(s); ce.append(e)
    assert covered == len(text), "pre-tokeniser did not cover the whole text"
    return ids, cs, ce


# ----------------------------------------------------------------------------------------------------
# 3. BIO tags <-> spans
# ----------------------------------------------------------------------------------------------------
IGNORE = -100


def tag_vocab(labels):
    tags = ["O"] + [f"{p}-{l}" for l in labels for p in ("B", "I")]
    return tags, {t: i for i, t in enumerate(tags)}


def bio_tags(cs, ce, spans, tag2id):
    """gold spans -> one tag per token. Whitespace-only tokens -> IGNORE (no loss, transparent when decoding).
    Returns tags and the number of overlapping (conflicting) spans that were skipped."""
    n = len(cs)
    tags = [IGNORE if cs[i] == ce[i] else 0 for i in range(n)]
    taken = [False] * n
    conflicts = 0
    for s, e, lab in sorted(spans, key=lambda x: (x[0], -(x[1] - x[0]))):
        lo = bisect.bisect_right(ce, s)               # first token whose core ends after s (ce is sorted)
        hi = bisect.bisect_left(cs, e, lo)            # first token whose core starts at/after e
        idx = [i for i in range(lo, hi) if cs[i] < ce[i]]
        if not idx:
            continue
        if any(taken[i] for i in idx):
            conflicts += 1
            continue
        for k, i in enumerate(idx):
            tags[i] = tag2id[("B-" if k == 0 else "I-") + lab]
            taken[i] = True
    return tags, conflicts


def decode_spans(tags, cs, ce, id2tag):
    """tag ids per token -> [(s, e, label)]. B-X starts a span; I-X continues a span of type X (an I-X
    after O or after another type also starts one: lenient repair); whitespace tokens are skipped."""
    spans, cur = [], None                             # cur = [label, start, end]
    for t, s, e in zip(tags, cs, ce):
        if s == e:                                    # whitespace: transparent
            continue
        tag = id2tag[t] if t >= 0 else "O"
        if tag == "O":
            if cur: spans.append(tuple(cur)); cur = None
            continue
        p, lab = tag.split("-", 1)
        if p == "B" or cur is None or cur[0] != lab:
            if cur: spans.append(tuple(cur))
            cur = [lab, s, e]
        else:
            cur[2] = e
    if cur: spans.append(tuple(cur))
    return [(s, e, lab) for lab, s, e in spans]


def windows(n, size, stride):
    """start positions of windows of `size` tokens covering [0, n); the last one is pulled back to end at n,
    so every window (except in documents shorter than size) has exactly `size` tokens - no padding."""
    if n <= size:
        return [0]
    starts = list(range(0, n - size + 1, stride))
    if starts[-1] + size < n:
        starts.append(n - size)
    return starts


# ----------------------------------------------------------------------------------------------------
# 4. Strict span scoring (exact start, end and label)
# ----------------------------------------------------------------------------------------------------
def strict_scores(pred, gold, labels):
    """pred/gold: {doc: [(s,e,label)]} -> dict with macro P/R/F1 and per-label rows."""
    tp, fp, fn = Counter(), Counter(), Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        for x in p & g: tp[x[2]] += 1
        for x in p - g: fp[x[2]] += 1
        for x in g - p: fn[x[2]] += 1
    rows = {}
    for l in labels:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        F = 2 * P * R / (P + R) if P + R else 0.0
        rows[l] = dict(P=P, R=R, F1=F, tp=tp[l], fp=fp[l], fn=fn[l], support=tp[l] + fn[l])
    used = [l for l in labels if rows[l]["support"] or rows[l]["fp"]]
    mac = lambda k: sum(rows[l][k] for l in used) / max(1, len(used))
    T, FP, FN = sum(tp.values()), sum(fp.values()), sum(fn.values())
    micro = 2 * T / (2 * T + FP + FN) if T else 0.0
    return dict(macro_F1=mac("F1"), macro_P=mac("P"), macro_R=mac("R"), micro_F1=micro, per_label=rows)


def error_breakdown(pred, gold):
    """Classify every mistake: wrong label on an exact span, boundary error (overlap, same label),
    boundary+label error, missed entity, spurious prediction."""
    c = Counter()
    for doc in set(gold) | set(pred):
        g = set(map(tuple, gold.get(doc, []))); p = set(map(tuple, pred.get(doc, [])))
        c["correct"] += len(p & g)
        gs = sorted(g - p); ps = sorted(p - g)
        used = set()
        for s, e, l in gs:
            over = [(ps_, pe_, pl) for ps_, pe_, pl in ps if ps_ < e and pe_ > s]
            if not over:
                c["missed"] += 1; continue
            used.update(over)
            if any(o[0] == s and o[1] == e for o in over): c["wrong_label_exact_span"] += 1
            elif any(o[2] == l for o in over): c["boundary_error_same_label"] += 1
            else: c["boundary_and_label_error"] += 1
        c["spurious"] += len([x for x in ps if x not in used])
    return dict(c)


# ----------------------------------------------------------------------------------------------------
# 5. Submission
# ----------------------------------------------------------------------------------------------------
def to_submission(pred, conv, texts=None):
    """{doc: [(s, e_exclusive, label)]} -> {doc: [[s, e, label, text]]} in the dataset's own end convention."""
    sub = 1 if conv == "inclusive" else 0
    out = {}
    for doc, spans in pred.items():
        rows = []
        for s, e, l in sorted(spans):
            row = [int(s), int(e - sub), l]
            if texts is not None: row.append(texts[doc][s:e])
            rows.append(row)
        out[doc] = rows
    return out

In [ ]:
%%writefile p2src/ner_model.py
# Phase 2 (L-NER) model side: load a Phase-1 backbone, put a token-classification head on it,
# train / predict with sliding windows.
import math
import os
import pickle
import random
import sys
import time

import torch
import torch.nn as nn
import torch.nn.functional as F

import ner_data as nd

ARCH_INFO = {   # folder -> (tokenizer file, longest window the backbone was trained on)
    "arch1_decoder": ("a1_bpe8000.pkl", 1024),
    "arch2_encdec": ("a1_bpe8000.pkl", 768),
    "arch3_prefixlm": ("space_bpe8000.pkl", 1024),
    "arch4_mla": ("space_bpe8000.pkl", 1024),
}


def autocast():
    return torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available())


# ---- MLA attention without the causal mask (only used when bidirectional=True and no KV cache) ----
def _mla_bidirectional_forward(self, x, cos, sin, cache=None):
    if cache is not None or not getattr(self, "bidirectional", False):
        return self._causal_forward(x, cos, sin, cache)
    from rope import apply_rope
    B, T, _ = x.shape
    q = self.q_proj(x).view(B, T, self.h, self.nope + self.rope).transpose(1, 2)
    q_nope, q_rope = q.split([self.nope, self.rope], dim=-1)
    q_rope = apply_rope(q_rope, cos.to(q.dtype), sin.to(q.dtype))
    c_kv = self.kv_norm(self.kv_down(x))
    k_r = apply_rope(self.k_rope(x)[:, None], cos.to(q.dtype), sin.to(q.dtype))
    k_nope = self.k_up(c_kv).view(B, T, self.h, self.nope).transpose(1, 2)
    v = self.v_up(c_kv).view(B, T, self.h, self.vd).transpose(1, 2)
    q = torch.cat([q_nope, q_rope], dim=-1)
    k = torch.cat([k_nope, k_r.expand(B, self.h, T, self.rope)], dim=-1)
    out = F.scaled_dot_product_attention(q, k, v, is_causal=False,
                                         dropout_p=self.dropout if self.training else 0.0)
    return self.o_proj(out.transpose(1, 2).reshape(B, T, -1)), None


class Backbone(nn.Module):
    """A Phase-1 model reduced to a feature extractor: token ids (B,T) -> hidden states (B,T,d)."""

    def __init__(self, arch, root, bidirectional=True):
        super().__init__()
        code = os.path.join(root, "code")
        if code not in sys.path:
            sys.path.insert(0, code)
        self.arch, self.bidirectional = arch, bidirectional
        ck = torch.load(os.path.join(root, arch, "ckpt_best.pt"), map_location="cpu", weights_only=False)
        cfg = dict(ck["cfg"])
        if arch == "arch4_mla":
            from mla import MLAConfig, MLALM, MLAttention
            if not hasattr(MLAttention, "_causal_forward"):
                MLAttention._causal_forward = MLAttention.forward
                MLAttention.forward = _mla_bidirectional_forward
            m = MLALM(MLAConfig(**cfg))
        elif arch == "arch3_prefixlm":
            from prefixlm import PrefixLMConfig, PrefixLM
            m = PrefixLM(PrefixLMConfig(**cfg))
        elif arch == "arch2_encdec":
            from encdec import EncDecConfig, EncDecLM
            m = EncDecLM(EncDecConfig(**cfg))
        elif arch == "arch1_decoder":
            from model_arch import LMConfig, DecoderLM
            m = DecoderLM(LMConfig(**cfg))
        else:
            raise ValueError(arch)
        missing, unexpected = m.load_state_dict(ck["model"], strict=False)
        assert not unexpected and all("rope" in k or "pos_emb" in k or "mask" in k for k in missing), \
            (missing[:5], unexpected[:5])
        self.phase1_step, self.phase1_val = ck.get("step"), ck.get("val_loss")
        if arch == "arch2_encdec":                        # NER needs only the (bidirectional) encoder
            m.decoder = nn.ModuleList(); m.dec_norm = nn.Identity()
        if arch == "arch1_decoder" and bidirectional:     # A3 attention keeps its causal mask in a buffer
            for blk in m.blocks:
                blk.attn.causal_mask.fill_(True)
        if arch == "arch4_mla":
            for blk in m.blocks:
                blk.attn.bidirectional = bidirectional
        self.m = m
        self.d = cfg["d_model"]
        self.max_len = ARCH_INFO[arch][1]
        self.lm_head_params = 0                           # the tied LM head is not used (no extra params)

    def forward(self, ids):
        m = self.m
        if self.arch == "arch4_mla":
            return m.hidden(ids)
        if self.arch == "arch3_prefixlm":                 # prefix = whole window -> fully bidirectional
            return m.hidden(ids, prefix_len=ids.size(1) if self.bidirectional else 0)
        if self.arch == "arch2_encdec":
            return m.encode(ids)
        B, T = ids.shape                                  # arch 1: DecoderLM.forward without the LM head
        x = m.tok_emb(ids)
        if m.cfg.scale_emb:
            x = x * math.sqrt(m.cfg.d_model)
        if m.cfg.pos_emb_type == "sinusoidal":
            x = x + m.pos_emb[:T].unsqueeze(0)
        for blk in m.blocks:
            x = blk(x)
        return m.ln_f(x)


def load_tokenizer(arch, root):
    code = os.path.join(root, "code")
    if code not in sys.path:
        sys.path.insert(0, code)
    import a1_tokenizer, space_bpe  # noqa: F401  (needed to unpickle)
    return pickle.load(open(os.path.join(root, "tokenizers", ARCH_INFO[arch][0]), "rb"))


class TokenTagger(nn.Module):
    """backbone -> dropout -> linear head over the BIO tags. forward(ids, tags) -> loss (1,), forward(ids) -> logits."""

    def __init__(self, backbone, n_tags, dropout=0.1, class_weights=None):
        super().__init__()
        self.backbone = backbone
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(backbone.d, n_tags)
        nn.init.normal_(self.head.weight, std=0.02); nn.init.zeros_(self.head.bias)
        self.register_buffer("class_weights", class_weights if class_weights is not None else torch.ones(n_tags))

    def forward(self, ids, tags=None):
        with autocast():
            h = self.backbone(ids)
            logits = self.head(self.drop(h)).float()
        if tags is None:
            return logits
        loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tags.reshape(-1),
                               weight=self.class_weights, ignore_index=nd.IGNORE)
        return loss.reshape(1)


# ----------------------------------------------------------------------------------------------------
# Data preparation: documents -> fixed-length windows
# ----------------------------------------------------------------------------------------------------
def prepare_docs(texts, gold, tok, tag2id, split="fine"):
    """{doc: text} (+ gold) -> {doc: dict(ids, cs, ce, tags)}; returns docs and alignment stats."""
    docs, conflicts = {}, 0
    for d, t in texts.items():
        ids, cs, ce = nd.encode_with_offsets(tok, t, split)
        rec = dict(ids=ids, cs=cs, ce=ce)
        if gold is not None:
            rec["tags"], c = nd.bio_tags(cs, ce, gold.get(d, []), tag2id); conflicts += c
        docs[d] = rec
    return docs, conflicts


def make_windows(docs, size, stride):
    out = []
    for d, r in docs.items():
        if not r["ids"]:
            continue
        for s in nd.windows(len(r["ids"]), size, stride):
            out.append((d, s, min(size, len(r["ids"]))))
    return out


def batches(win, bs, shuffle, rng=None):
    """group windows of equal length (no padding needed), then cut into batches"""
    by_len = {}
    for w in win:
        by_len.setdefault(w[2], []).append(w)
    out = []
    for L, ws in by_len.items():
        if shuffle: rng.shuffle(ws)
        out += [ws[i:i + bs] for i in range(0, len(ws), bs)]
    if shuffle: rng.shuffle(out)
    return out


def _tensor(docs, batch, key, device):
    return torch.tensor([docs[d][key][s:s + L] for d, s, L in batch], dtype=torch.long, device=device)


@torch.no_grad()
def predict(net, docs, size, stride, bs, device, id2tag):
    """sliding-window inference; overlapping windows are averaged with a tent weight (centre counts most)."""
    net.eval()
    acc = {d: None for d in docs}
    win = make_windows(docs, size, stride)
    for b in batches(win, bs, shuffle=False):
        logits = net(_tensor(docs, b, "ids", device)).float().cpu()
        L = logits.size(1)
        w = (1 + torch.minimum(torch.arange(L), torch.arange(L).flip(0))).float()[:, None]
        for (d, s, _), lg in zip(b, logits):
            if acc[d] is None:
                n = len(docs[d]["ids"]); acc[d] = [torch.zeros(n, lg.size(-1)), torch.zeros(n, 1)]
            acc[d][0][s:s + L] += lg.softmax(-1) * w; acc[d][1][s:s + L] += w
    pred = {}
    for d, v in acc.items():
        if v is None:                                     # empty document
            pred[d] = []; continue
        p, w = v
        tags = (p / w).argmax(-1).tolist()
        pred[d] = nd.decode_spans(tags, docs[d]["cs"], docs[d]["ce"], id2tag)
    return pred


def trainable_params(net):
    return sum(p.numel() for p in net.parameters() if p.requires_grad)


def train(net, train_docs, val_docs, val_gold, labels, id2tag, cfg, device, log=print, on_epoch=None):
    """Fine-tune with AdamW (separate LR for backbone and head), linear warm-up + linear decay, fp16,
    DataParallel over all GPUs. Keeps the weights of the best validation macro-F1 epoch."""
    rng = random.Random(cfg["seed"])
    gpus = torch.cuda.device_count()
    para = nn.DataParallel(net) if gpus > 1 else net
    head_names = ("head.",)
    groups = [
        {"params": [p for n, p in net.named_parameters() if p.requires_grad and not n.startswith(head_names)],
         "lr": cfg["lr_backbone"]},
        {"params": [p for n, p in net.named_parameters() if p.requires_grad and n.startswith(head_names)],
         "lr": cfg["lr_head"]},
    ]
    groups = [g for g in groups if g["params"]]
    for g in groups: g["base_lr"] = g["lr"]
    opt = torch.optim.AdamW(groups, weight_decay=cfg["weight_decay"])
    scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())
    win = make_windows(train_docs, cfg["window"], cfg["train_stride"])
    steps_per_epoch = math.ceil(len(win) / cfg["batch_size"])
    total = steps_per_epoch * cfg["epochs"]; warm = int(cfg["warmup_frac"] * total)
    log(f"train windows {len(win):,} | {steps_per_epoch} steps/epoch x {cfg['epochs']} epochs | "
        f"trainable params {trainable_params(net):,} | GPUs {max(gpus, 1)}")
    best, best_state, history, step, t0 = -1.0, None, [], 0, time.time()
    if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
    for ep in range(1, cfg["epochs"] + 1):
        net.train(); run, n = 0.0, 0
        for b in batches(win, cfg["batch_size"], shuffle=True, rng=rng):
            step += 1
            f = step / max(1, warm) if step <= warm else max(0.0, (total - step) / max(1, total - warm))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            loss = para(_tensor(train_docs, b, "ids", device), _tensor(train_docs, b, "tags", device)).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_([p for g in groups for p in g["params"]], cfg["clip"])
            scaler.step(opt); scaler.update()
            run += loss.item(); n += 1
            if step % cfg["log_every"] == 0:
                log(f"  ep {ep} step {step}/{total} | loss {run / n:.4f} | {(time.time() - t0) / 60:.1f} min")
                run, n = 0.0, 0
            if (time.time() - t0) / 3600 > cfg["max_train_hours"]:
                break
        pred = predict(net, val_docs, cfg["window"], cfg["eval_stride"], cfg["eval_batch"], device, id2tag)
        sc = nd.strict_scores(pred, val_gold, labels)
        history.append(dict(epoch=ep, step=step, macro_F1=sc["macro_F1"], micro_F1=sc["micro_F1"],
                            minutes=round((time.time() - t0) / 60, 1)))
        note = ""
        if sc["macro_F1"] > best:
            best = sc["macro_F1"]; note = "  <- best"
            best_state = {k: v.detach().to("cpu", copy=True) for k, v in net.state_dict().items()}
        log(f"epoch {ep}: val strict macro-F1 {sc['macro_F1']:.4f} | micro-F1 {sc['micro_F1']:.4f} "
            f"| P {sc['macro_P']:.3f} R {sc['macro_R']:.3f}{note}")
        if on_epoch: on_epoch(ep, sc)
        if (time.time() - t0) / 3600 > cfg["max_train_hours"]:
            log("time budget reached - stopping"); break
    net.load_state_dict(best_state)
    peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0.0
    return dict(best_macro_F1=best, history=history, train_minutes=round((time.time() - t0) / 60, 1),
                peak_gpu_gb=round(peak, 2), trainable_params=trainable_params(net))

In [ ]:
import importlib
sys.path.insert(0, os.path.abspath("p2src"))
import ner_data as nd, ner_model as nm
importlib.reload(nd); importlib.reload(nm)
print("Phase-2 code imported")

In [ ]:
# 4 · Load Phase_2_LNER
from huggingface_hub import HfApi, hf_hub_download
files = [f for f in HfApi().list_repo_files(CFG["hf_repo"], repo_type="dataset") if f.startswith("Phase_2")]
print("\n".join(files))
get = lambda name: hf_hub_download(CFG["hf_repo"], next(f for f in files if f.endswith(name)), repo_type="dataset")
load = lambda name: json.load(open(get(name), encoding="utf-8"))

LABELS = [l.strip() for l in open(get("labels.txt"), encoding="utf-8") if l.strip()]
texts, gold_raw = {}, {}
for fold in ("fold1", "fold2"):
    t = nd.parse_texts(load(f"{fold}-text.json")); g = nd.parse_gold(load(f"{fold}-gold.json"))
    print(f"{fold}: {len(t)} documents, {sum(map(len, g.values())):,} entities, "
          f"{sum(map(len, t.values())) / max(1, len(t)):,.0f} chars/doc on average")
    assert set(g) <= set(t), "gold ids without a document"
    texts.update(t); gold_raw.update({d: g.get(d, []) for d in t})
test_texts = nd.parse_texts(load("fold3-text.json"))
print(f"test (fold3): {len(test_texts)} documents")

CONV, info = nd.end_convention(texts, gold_raw)
print(f"\nend offset convention: {CONV}  {({k: v for k, v in info.items() if k != 'examples_no_match'})}")
for ex in info["examples_no_match"]: print("   no match:", ex)
GOLD = nd.to_exclusive(gold_raw, CONV)

from collections import Counter
cnt = Counter(l for ents in GOLD.values() for _, _, l in ents)
unknown = set(cnt) - set(LABELS)
assert not unknown, f"labels in gold but not in labels.txt: {unknown}"
print("\nlabels:", LABELS)
for l in LABELS:
    ex = next((texts[d][s:e] for d, ents in GOLD.items() for s, e, ll in ents if ll == l), "")
    print(f"  {l:<10} {cnt[l]:>6,}   e.g. {ex[:60]!r}")

In [ ]:
# 5 · Document-level validation split (whole documents, so no text leaks between train and val)
ids = sorted(texts); random.Random(CFG["seed"]).shuffle(ids)
n_val = max(1, int(round(CFG["val_frac"] * len(ids))))
VAL_IDS, TRAIN_IDS = sorted(ids[:n_val]), sorted(ids[n_val:])
if CFG["smoke"]: TRAIN_IDS, VAL_IDS = TRAIN_IDS[:20], VAL_IDS[:5]
val_gold = {d: GOLD[d] for d in VAL_IDS}
c_tr = Counter(l for d in TRAIN_IDS for *_, l in GOLD[d]); c_va = Counter(l for d in VAL_IDS for *_, l in GOLD[d])
print(f"train {len(TRAIN_IDS)} docs | val {len(VAL_IDS)} docs")
print("val entities per label:", {l: c_va[l] for l in LABELS})

In [ ]:
# 6 · Offset round-trip check: gold spans -> token tags -> spans. This is the best score any token
#     tagger can reach with this tokenisation (strict matching!). Compared for both pre-tokenisations.
TAGS, TAG2ID = nd.tag_vocab(LABELS); ID2TAG = dict(enumerate(TAGS))
tok = nm.load_tokenizer(CFG["arch"], ROOT)
check_ids = TRAIN_IDS[:200]
ALIGN = {}
for split in ("bpe", "fine"):
    t0 = time.time(); rec, n_tok, conf = {}, 0, 0
    for d in check_ids:
        ids_, cs, ce = nd.encode_with_offsets(tok, texts[d], split)
        assert tok.decode(ids_) == texts[d]
        tags, c = nd.bio_tags(cs, ce, GOLD[d], TAG2ID); conf += c; n_tok += len(ids_)
        rec[d] = nd.decode_spans(tags, cs, ce, ID2TAG)
    sc = nd.strict_scores(rec, {d: GOLD[d] for d in check_ids}, LABELS)
    ALIGN[split] = dict(ceiling_macro_F1=round(sc["macro_F1"], 4), ceiling_micro_F1=round(sc["micro_F1"], 4),
                        tokens_per_doc=round(n_tok / len(check_ids)), overlapping_spans_skipped=conf)
    print(f"{split:>4}: ceiling strict macro-F1 {sc['macro_F1']:.4f} | micro {sc['micro_F1']:.4f} | "
          f"{n_tok / len(check_ids):,.0f} tokens/doc | overlapping gold spans skipped {conf} | {time.time()-t0:.0f}s")
    if split == CFG["split"]:
        lost = [(texts[d][s:e], l) for d in check_ids for s, e, l in sorted(set(GOLD[d]) - set(rec[d]))]
        print("   examples that cannot be cut exactly:", lost[:8])
print("-> using split =", CFG["split"])

In [ ]:
# 7 · Tokenise all documents, build the model
t0 = time.time()
train_docs, conf = nm.prepare_docs({d: texts[d] for d in TRAIN_IDS}, GOLD, tok, TAG2ID, CFG["split"])
val_docs, _ = nm.prepare_docs({d: texts[d] for d in VAL_IDS}, GOLD, tok, TAG2ID, CFG["split"])
test_docs, _ = nm.prepare_docs(test_texts, None, tok, TAG2ID, CFG["split"])
print(f"tokenised in {time.time()-t0:.0f}s | train tokens {sum(len(r['ids']) for r in train_docs.values()):,}")

tag_counts = Counter(t for r in train_docs.values() for t in r["tags"] if t >= 0)
weights = None
if CFG["class_weighting"] == "sqrt_inv":
    tot = sum(tag_counts.values())
    weights = torch.tensor([math.sqrt(tot / max(1, tag_counts[i])) for i in range(len(TAGS))])
    weights = weights / weights[0]
print("share of O tags:", round(tag_counts[0] / sum(tag_counts.values()), 4))

backbone = nm.Backbone(CFG["arch"], ROOT, bidirectional=CFG["bidirectional"])
assert CFG["window"] <= backbone.max_len
net = nm.TokenTagger(backbone, len(TAGS), CFG["dropout"], weights).to(DEVICE)
total = sum(p.numel() for p in net.parameters())
print(f"backbone {CFG['arch']} (Phase-1 step {backbone.phase1_step}, val loss {backbone.phase1_val}) | "
      f"total params {total:,} | trainable {nm.trainable_params(net):,}")

In [ ]:
# 8 · Fine-tune (best validation epoch is kept)
result = nm.train(net, train_docs, val_docs, val_gold, LABELS, ID2TAG, CFG, DEVICE)
print(f"best val strict macro-F1 {result['best_macro_F1']:.4f} | {result['train_minutes']} min | "
      f"peak GPU memory {result['peak_gpu_gb']} GB")

In [ ]:
# 9 · Validation analysis: per label + kind of error
t0 = time.time()
val_pred = nm.predict(net, val_docs, CFG["window"], CFG["eval_stride"], CFG["eval_batch"], DEVICE, ID2TAG)
infer_sec = time.time() - t0
VAL = nd.strict_scores(val_pred, val_gold, LABELS)
print(f"strict macro-F1 {VAL['macro_F1']:.4f} | macro-P {VAL['macro_P']:.4f} | macro-R {VAL['macro_R']:.4f} | "
      f"micro-F1 {VAL['micro_F1']:.4f} | inference {infer_sec:.0f}s for {len(val_docs)} docs")
print(f"\n{'label':<10} {'P':>6} {'R':>6} {'F1':>6} {'support':>8}")
for l in LABELS:
    r = VAL["per_label"][l]; print(f"{l:<10} {r['P']:6.3f} {r['R']:6.3f} {r['F1']:6.3f} {r['support']:8d}")
ERR = nd.error_breakdown(val_pred, val_gold); print("\nerrors:", ERR)
d0 = VAL_IDS[0]; g0, p0 = set(val_gold[d0]), set(val_pred[d0])
print("\nexamples (gold not predicted):", [(texts[d0][s:e], l) for s, e, l in sorted(g0 - p0)][:6])
print("examples (predicted, wrong):  ", [(texts[d0][s:e], l) for s, e, l in sorted(p0 - g0)][:6])

In [ ]:
# 10 · Test predictions -> submission zip, record, fine-tuned weights
t0 = time.time()
test_pred = nm.predict(net, test_docs, CFG["window"], CFG["eval_stride"], CFG["eval_batch"], DEVICE, ID2TAG)
sub = nd.to_submission(test_pred, CONV)
assert set(sub) == set(test_texts)
for d, rows in sub.items():
    for s, e, l in rows: assert l in LABELS and 0 <= s <= e <= len(test_texts[d])
pj = os.path.join(RUN_DIR, "prediction.json"); json.dump(sub, open(pj, "w"), ensure_ascii=False)
zp = os.path.join(RUN_DIR, f"submission_{CFG['run']}.zip")
with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
with zipfile.ZipFile(zp) as z: assert z.namelist() == ["prediction.json"]
print(f"{sum(map(len, sub.values())):,} entities in {len(sub)} test docs ({time.time()-t0:.0f}s) -> {zp}")
print("predicted labels:", Counter(l for rows in sub.values() for *_, l in rows))

record = dict(cfg=CFG, end_convention=CONV, alignment=ALIGN, n_train_docs=len(TRAIN_IDS), n_val_docs=len(VAL_IDS),
              total_params=total, **result, val=dict(macro_F1=VAL["macro_F1"], macro_P=VAL["macro_P"],
              macro_R=VAL["macro_R"], micro_F1=VAL["micro_F1"], per_label=VAL["per_label"]),
              errors=ERR, val_inference_sec=round(infer_sec, 1), zip=os.path.basename(zp))
json.dump(record, open(os.path.join(RUN_DIR, "record.json"), "w"), indent=1)
torch.save({"model": net.state_dict(), "arch": CFG["arch"], "tags": TAGS, "cfg": CFG},
           os.path.join(RUN_DIR, "ner_best.pt"))
print("saved record.json + ner_best.pt in", RUN_DIR)
if CFG["smoke"]: print("SMOKE RUN - do not submit this zip")